## ДЗ №2
Построить и оценить качество бейзлайна

Повторим ещё раз информация о датасете:

Бизнес-постановка задачи: Нужно предсказать какие клиенты телеком-компании собираются уйти, чтобы предложить им индивидуальные акции, различные целенеправленные программы удержания, чтобы удержать клиента/снизить отток.

ML постановка задачи: Задача бинарной классификации, где 1-клиент уйдёт, 0-останется

Набор данных: Источник: Kaggle - https://www.kaggle.com/datasets/blastchar/telco-customer-churn/data

В датасете - 7043 строк, 21 столбец, из них 19 признаков. Исходный столбец "Churn" наша целевая метка

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv("Telco-Customer-Churn.csv")

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df = df.drop("customerID",axis=1)

В прошлом ДЗ №1 было выявлено, что колонка TotalCharges имеет 11 пропущенных значений. Повторим ещё раз эти действия

In [5]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].dtypes
df["TotalCharges"].isna().sum()

11

Приведя наш признак из типа object в числовой и обработав различные ошибки с помощью аргумента errors="coerce" в Nan, а после снова проверив на различные пропущенные значения, мы получили 11 пропущенных значений NaN в столбце TotalCharges

In [6]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7032.000000
mean,0.162147,32.371149,64.761692,2283.300441
std,0.368612,24.559481,30.090047,2266.771362
min,0.000000,0.000000,18.250000,18.800000
25%,0.000000,9.000000,35.500000,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.850000,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


Выведя информацию о числовых признаках, можно сделать вывод, что SeniorCitizen это просто бинарный категориальный признак 0/1 пожилой гражданин или нет.

In [7]:
for i in df.columns:
  print(i, df[i].unique())

gender ['Female' 'Male']
SeniorCitizen [0 1]
Partner ['Yes' 'No']
Dependents ['No' 'Yes']
tenure [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
PhoneService ['No' 'Yes']
MultipleLines ['No phone service' 'No' 'Yes']
InternetService ['DSL' 'Fiber optic' 'No']
OnlineSecurity ['No' 'Yes' 'No internet service']
OnlineBackup ['Yes' 'No' 'No internet service']
DeviceProtection ['No' 'Yes' 'No internet service']
TechSupport ['No' 'Yes' 'No internet service']
StreamingTV ['No' 'Yes' 'No internet service']
StreamingMovies ['No' 'Yes' 'No internet service']
Contract ['Month-to-month' 'One year' 'Two year']
PaperlessBilling ['Yes' 'No']
PaymentMethod ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
MonthlyCharges [29.85 56.95 53.85 ... 63.1  44.2  78.7 ]
TotalCharges [  29.85 1889.

Взглянем на уникальные значения наших столбцов

# Определим какое кодирование принимать к признакам:
# 1) Признаки gender, SeniorCitizen, Partner, Dependents, PhoneService, PaperlessBilling являются бинарными категориальными без порядка переводим их в Yes > 1 , No > 0

# 2) Признаки MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies,PaymentMethod - категориальные номинальные, поэтому так как мы будем использовать дерево решений, то используем Target Encoding(а не в one-hot)

# 3) Признак - contract - порядковый, поэтому используем Ordinalencoding с указанием порядка, что month = 0, year = 1, two year = 2

# 4) Признаки tenure, MonthlyCharges, TotalCharges - числовые

In [8]:
df.columns = df.columns.str.strip()

Удалили пробелы и другие невидимые символы

In [9]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

Преобразовали нашу целевую переменную в числовой вид, где Yes = 1, No = 0

In [10]:
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y
)

Разделим наш набор данных на train и test 80/20 со стратификацией

Мы имеет числовой признак TotalCharges, который имеет пропуски и который по исследованию в ДЗ №1 имеет  очень огромный разброс относительно среднего и большое смещение среднего вправо относительно медианы. В таком случае лучше заменять пропущенные значения медианой, а не средним.

In [11]:
median_total = X_train['TotalCharges'].median()
X_train['TotalCharges'] = X_train['TotalCharges'].fillna(median_total)
X_test['TotalCharges'] = X_test['TotalCharges'].fillna(median_total)

Заменили вычисленной медианой из обучающей выборки, так как нельзя вычислять медиану по тестовой для тестовой, ведь тогда мы как бы заглянем в данные, о которых не знаем

Начнём кодирование признаков:

In [12]:
binary_cols = ['gender', 'Partner', 'Dependents',
               'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    if X_train[col].dtype == 'object' and col != 'gender':
        X_train[col] = X_train[col].map({'Yes': 1, 'No': 0})
        X_test[col] = X_test[col].map({'Yes': 1, 'No': 0})
    elif X_train[col].dtype == 'object' and col == 'gender':
        X_train[col] = X_train[col].map({'Male': 1, 'Female': 0})
        X_test[col] = X_test[col].map({'Male': 1, 'Female': 0})


Произвели кодирование бинарных категориальных переменных

In [13]:
from sklearn.preprocessing import OrdinalEncoder

years = ['Month-to-month', 'One year', 'Two year']
encoder = OrdinalEncoder(categories=[years])

X_train[['Contract']] = encoder.fit_transform(X_train[['Contract']])

X_test[['Contract']] = encoder.transform(X_test[['Contract']])

Произвели кодирование порядкового категориального признака

In [14]:
from sklearn.preprocessing import TargetEncoder


cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport',  'StreamingTV', 'StreamingMovies', 'PaymentMethod']

encoder = TargetEncoder(smooth='auto')
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols], y_train)
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

Провели кодирование категориальных номинальных признаков

In [15]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5634 entries, 3738 to 5639
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            5634 non-null   int64  
 1   SeniorCitizen     5634 non-null   int64  
 2   Partner           5634 non-null   int64  
 3   Dependents        5634 non-null   int64  
 4   tenure            5634 non-null   int64  
 5   PhoneService      5634 non-null   int64  
 6   MultipleLines     5634 non-null   float64
 7   InternetService   5634 non-null   float64
 8   OnlineSecurity    5634 non-null   float64
 9   OnlineBackup      5634 non-null   float64
 10  DeviceProtection  5634 non-null   float64
 11  TechSupport       5634 non-null   float64
 12  StreamingTV       5634 non-null   float64
 13  StreamingMovies   5634 non-null   float64
 14  Contract          5634 non-null   float64
 15  PaperlessBilling  5634 non-null   int64  
 16  PaymentMethod     5634 non-null   float64
 1

Таким образом все признаки имеют теперь численные значения

In [16]:
X_train

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
3738,1,0,0,0,35,0,0.236016,0.184369,0.418779,0.401464,0.228574,0.417681,0.304840,0.301053,0.0,0,0.458583,49.20,1701.65
3151,1,0,1,1,15,1,0.252076,0.423405,0.143800,0.401464,0.388953,0.417681,0.331211,0.334989,0.0,0,0.190206,75.10,1151.55
4860,1,0,1,1,13,0,0.225526,0.179663,0.149011,0.214404,0.389398,0.148037,0.330285,0.335529,2.0,0,0.200441,40.55,590.35
3867,0,0,1,0,26,1,0.252076,0.184369,0.418779,0.216818,0.228574,0.417681,0.304840,0.301053,2.0,1,0.146419,73.50,1905.70
3810,1,0,1,1,1,1,0.250356,0.188305,0.422698,0.403964,0.387022,0.412446,0.324048,0.337228,0.0,0,0.453620,44.55,44.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6303,0,0,1,0,71,1,0.287580,0.416697,0.420111,0.218677,0.223319,0.144645,0.300602,0.304765,2.0,0,0.464854,109.25,7707.70
6227,1,0,0,0,2,1,0.248729,0.191281,0.420111,0.398382,0.391773,0.420894,0.334975,0.331027,0.0,0,0.161628,46.05,80.35
4673,0,1,0,0,25,1,0.287580,0.416697,0.146583,0.218677,0.391773,0.420894,0.300602,0.304765,0.0,1,0.189117,102.80,2660.20
2710,0,0,1,0,24,1,0.252076,0.074788,0.074788,0.074788,0.074788,0.074788,0.074788,0.074788,1.0,0,0.146419,20.40,482.80


Сделаем константное предсказание по самому частому классу в тренировочных данных

In [17]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print(f"F1-score:  {f1_score(y_test, y_pred_dummy):.3f}")
print(f"Recall:   {recall_score(y_test,y_pred_dummy):.3f}")

F1-score:  0.000
Recall:   0.000


Наши метрики Recall, F1 - получились равными нулю. Это ожидаемо так как класс 0 (клиент остался) встречается чаще -> DummyClassifier всегда предсказывает 0 и никогда не предсказывает уходящего клиента

Используем дерево решений. Мы можем не делать масштабирование признаков так как для деревьев решений оно не требуется.

In [18]:
tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42,
    class_weight='balanced'
)

tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

print(f"F1-score:  {f1_score(y_test, y_pred_tree):.3f}")
print(f"Recall:   {recall_score(y_test,y_pred_tree):.3f}")

F1-score:  0.622
Recall:   0.759


При всей несбалансированности наших классов, учитывая, что уходящих клиентов меньше, чем тех, которые остаются, модель показала неплохие метрики, на которые мы ориентируемся, особенно если сравнивать с DummyClassifer.

Если смотреть на метрику Recall, то можно сказать, что модель верно находит почти 76% уходящих клиентов, то есть, например, из 100 уходящих клиентов 76 из них модель предсказывает верно как уходящих